[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/Datacube-demo/blob/main/ODC/notebooks/01_odc_load_demo.ipynb)

# Open Data Cube   
The Open Data Cube (ODC) is an open source solution for accessing, managing, and analyzing large quantities of Geographic Information System (GIS) data - namely Earth Observation (EO) data. It presents a common analytical framework composed of a series of data structures and tools which facilitate the organization and analysis of large gridded data collections.

Open Data Cube（ODC）為開源之地球觀測資料管理與分析框架，其核心概念為：
**將散落之影像檔案建立索引，組織為可依產品名稱、時間與空間範圍直接查詢的資料立方（Data Cube）**。

分析端無須知悉檔案存放位置、檔名或座標系統，僅需提交查詢條件
（產品、空間範圍、時間區間），ODC 即自動完成相關檔案之篩選、裁切、重投影，
並堆疊為具時間維度之陣列（xarray）供後續分析使用。

### 教學內容

1. 於 Colab 環境建立完整之 ODC 執行環境（PostgreSQL 索引資料庫 + `datacube` 套件）。
2. 匯入 4 個年度之台灣 Sentinel-2 土地覆蓋（land cover）GeoTIFF 並建立索引。
3. 以 `dc.load()` 查詢大台北地區資料，進行跨年度地覆面積統計與地圖視覺化。

本教學旨在呈現 ODC 之核心特性：**分析程式碼僅描述所需資料之條件，
不涉及資料存放與讀取之細節**；相同程式碼適用於任意規模之資料量。

ODC 之系統架構與生態系可參考官方文件：[Open Data Cube Overview](https://www.opendatacube.org/overview-draft)

## ODC Sentinel-2 Land-Cover Load Demo

This notebook loads a fixed Greater Taipei bounding box from the indexed Taiwan Sentinel-2-derived annual land-cover GeoTIFFs.

`x` and `y` are projected raster coordinates in EPSG:32651. The plot below relabels ticks as longitude and latitude for readability. The land-cover class codes are stored in the `classification` data variable. NoData is class `0`.

### 示範資料說明

- **內容**：由 Sentinel-2 衛星影像衍生之台灣**年度土地覆蓋分類圖**，共 4 個年度（2017–2020）。
- **座標系統**：EPSG:32651（UTM Zone 51N，單位為公尺），解析度 10 公尺。
- **像元值**：土地覆蓋類別碼（1 水體、2 樹林、5 農作、7 建成區等），`0` 為無資料（NoData）。

後續圖表之 `x`、`y` 軸為投影座標，繪圖時將刻度轉換為經緯度以利判讀。

## Colab Setup

This section prepares a self-contained ODC environment inside the Colab VM:

1. Install and start PostgreSQL, then create the `datacube` database and user.
2. Install the `datacube` Python package (same version as the local Docker demo).
3. Clone this repository — Git LFS pulls the demo GeoTIFFs (~370 MB, takes a few minutes).
4. Initialize the ODC schema, add the `s2_landcover_taiwan` product, and index the demo datasets.

The full setup takes roughly 3–5 minutes on a fresh Colab runtime.

All setup cells are guarded by `IN_COLAB`, so they are safe no-ops when this notebook
runs in the local Docker environment (`odc_local_demo`), where the index is already
built by `setup_odc_demo.sh`.

### 中文說明

一個 ODC 環境由三個元件組成：**PostgreSQL 資料庫**（儲存索引與 metadata）、
**`datacube` Python 套件**（查詢與載入介面）、以及**實體資料檔**（GeoTIFF）。
正式環境中，上述元件由伺服器統一管理，僅需建置一次；Colab 環境則於每次
runtime 啟動時重新建置，全程約需 3–5 分鐘。各步驟與正式環境之對應如下：

| 步驟 | 動作 | 對應正式環境 |
| --- | --- | --- |
| 1 | 安裝並啟動 PostgreSQL、建立資料庫 | 資料庫伺服器（常駐） |
| 2 | 安裝 `datacube` 套件 | 分析環境 |
| 3 | 下載示範 GeoTIFF（Git LFS，約 370 MB） | 資料儲存區 |
| 4 | 初始化 schema、註冊產品、建立索引 | 資料匯入流程（僅執行一次） |

### Step 0：偵測執行環境

判斷目前執行環境是否為 Colab。後續建置步驟均以 `if IN_COLAB:` 條件保護；
於本地 Docker 環境（`odc_local_demo`）執行本 notebook 時，該等步驟將自動跳過
（環境已由 `setup_odc_demo.sh` 建置完成）。

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

### Step 1：安裝系統依賴

ODC 之索引儲存於 PostgreSQL，記錄每筆資料之時空範圍、座標系統與檔案路徑等
metadata。本步驟安裝並啟動資料庫，建立名為 `datacube` 之資料庫與使用者；
`git-lfs` 為下載示範 GeoTIFF 所需之工具。最後安裝 `datacube` 套件，
版本與本 repo 之 Docker 示範環境一致，以確保行為相同。

> 安裝過程中出現之 pip 相依性警告（如 SQLAlchemy 降版）屬預期現象，不影響後續執行。

In [ ]:
if IN_COLAB:
    # PostgreSQL backs the ODC index; git-lfs is needed to pull the demo GeoTIFFs.
    !sudo apt-get -qq update
    !sudo apt-get -qq install -y postgresql git-lfs
    !sudo service postgresql start
    !sudo -u postgres psql -tc "SELECT 1 FROM pg_roles WHERE rolname='datacube'" | grep -q 1 || sudo -u postgres psql -c "CREATE USER datacube WITH PASSWORD 'datacube';"
    !sudo -u postgres psql -tc "SELECT 1 FROM pg_database WHERE datname='datacube'" | grep -q 1 || sudo -u postgres createdb -O datacube datacube
    %pip install -q datacube==1.8.19 psycopg2-binary==2.9.9 pyyaml

### Step 2：下載資料並建立索引

本步驟為資料匯入 ODC 之標準流程，涉及兩個核心概念：

- **Product（產品）**：一「類」資料之定義，包含名稱、波段（measurement）、資料型別、
  NoData 值等。本教學註冊之產品為 `s2_landcover_taiwan`。
- **Dataset（資料集）**：產品之下的一筆實體資料，通常對應一個時間切片之檔案。
  4 個年度之 GeoTIFF 對應 4 個 datasets。

執行順序：`datacube system init` 建立資料庫 schema → `datacube product add` 註冊產品定義
→ 為各 GeoTIFF 產生描述其時空範圍之 YAML → `datacube dataset add` 寫入索引。
**索引僅記錄 metadata 與檔案路徑，不複製資料本身。**

> 技術註記：`system init` 過程中須建立資料庫角色（DB roles），需較高權限，
> 故以 `postgres` 超級使用者執行，完成後再將 `agdc` schema 之權限授予 `datacube` 使用者。

In [ ]:
if IN_COLAB:
    import os

    REPO_DIR = "/content/Datacube-demo"
    DEMO_ROOT = f"{REPO_DIR}/ODC/odc_local_demo"

    if not os.path.exists(REPO_DIR):
        !git lfs install --skip-repo
        !git clone https://github.com/NTU-CompHydroMet-Lab/Datacube-demo.git {REPO_DIR}

    # Same connection settings that setup_odc_demo.sh writes inside the Docker container.
    # Write the config to a known, universally accessible location like /tmp.
    DATACUBE_CONFIG_FILE = "/tmp/.datacube.conf"
    with open(DATACUBE_CONFIG_FILE, "w") as f:
        f.write(
            "[datacube]\n"
            "db_hostname: localhost\n"
            "db_database: datacube\n"
            "db_username: datacube\n"
            "db_password: datacube\n"
        )

    # Set the DATACUBE_CONFIG_PATH environment variable for Python processes
    # and for the shell commands that follow.
    os.environ['DATACUBE_CONFIG_PATH'] = DATACUBE_CONFIG_FILE

    # Execute datacube commands as the postgres system user.
    # Pass DATACUBE_CONFIG_PATH as an environment variable to each command executed by sudo.

    # Run datacube system init as the postgres system user,
    # and override the DB_USERNAME environment variable to 'postgres'
    # so it connects with superuser privileges to create the schema.
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} DB_USERNAME=postgres datacube system init

    # Grant privileges to the 'datacube' user on the 'agdc' schema and its objects.
    # This ensures the 'datacube' user can access objects created by 'postgres' user during init.
    !sudo -u postgres psql -d datacube -c "GRANT USAGE ON SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "ALTER DEFAULT PRIVILEGES IN SCHEMA agdc GRANT ALL PRIVILEGES ON TABLES TO datacube;"
    !sudo -u postgres psql -d datacube -c "GRANT ALL PRIVILEGES ON ALL SEQUENCES IN SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "ALTER DEFAULT PRIVILEGES IN SCHEMA agdc GRANT ALL PRIVILEGES ON SEQUENCES TO datacube;"

    # Verify system setup before proceeding.
    # Other commands can connect as the 'datacube' DB user, so no DB_USERNAME override is needed.
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube system check

    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube product add {DEMO_ROOT}/products/s2_landcover_taiwan.yaml
    # The python script itself doesn't need the datacube config path, as it's not calling datacube CLI directly.
    !python {DEMO_ROOT}/scripts/write_dataset_yaml.py --data-dir {DEMO_ROOT}/data --dataset-dir {DEMO_ROOT}/datasets --product s2_landcover_taiwan --measurement classification
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube dataset add {DEMO_ROOT}/datasets/*.yaml

### Step 3：驗證環境

以 repo 內建之檢查腳本驗證建置結果：產品已註冊、4 個 datasets 已入索引、
抽樣載入成功。輸出顯示 `OK` 即表示資料立方已就緒。

In [ ]:
if IN_COLAB:
    # Sanity check: product indexed, datasets found, sample load succeeds.
    !DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} python {DEMO_ROOT}/scripts/check_odc_demo.py

## 連線資料立方

環境建置完成後，分析端之唯一入口為 `Datacube` 物件。該物件依設定檔
（或環境變數）連線至 PostgreSQL 索引，後續之查詢與載入均經由此物件執行。
**自此以下之程式碼與正式部署環境完全相同**，不因底層環境
（Colab、本地 Docker 或伺服器）而異。

In [ ]:
from datacube import Datacube
from rasterio.warp import transform, transform_bounds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

PRODUCT = "s2_landcover_taiwan"
MEASUREMENT = "classification"
OUTPUT_CRS = "EPSG:32651"
RESOLUTION = (-10, 10)

# On Colab, Datacube() picks up the DATACUBE_CONFIG_PATH env var set in the setup cell.
dc = Datacube(app="01_odc_load_demo")

### 查詢產品清單

`dc.list_products()` 列出索引中所有已註冊之產品及其定義，
為探索資料立方內容之起點。

In [ ]:
dc.list_products().loc[[PRODUCT]]

### 查詢產品之資料集

`dc.index.datasets.search()` 回傳指定產品下所有已索引之資料集。
此處應回傳 **4 筆**，對應 2017–2020 四個年度之土地覆蓋圖；
每筆記錄包含時間範圍、空間範圍與實體檔案位置。

In [ ]:
datasets = list(dc.index.datasets.search(product=PRODUCT))
len(datasets), datasets[:1]

## Taipei Bounding Box

The bbox below is a focused Taipei core area in lon/lat. It is transformed to EPSG:32651 before loading from ODC.

### 中文說明

查詢範圍以經緯度（EPSG:4326）定義大台北核心區之範圍框，
於查詢前以 `transform_bounds` 轉換為資料之投影座標系（EPSG:32651）。
此為使用投影資料之標準前置步驟：範圍慣以經緯度描述，而資料以公尺網格儲存，
查詢條件須先換算至資料之座標系。

In [ ]:
# Greater Taipei core bbox: west, south, east, north in EPSG:4326
# Use a focused bbox for an interactive notebook; expand only when needed.
taipei_bbox_lonlat = (121.40, 24.95, 121.70, 25.20)

min_x, min_y, max_x, max_y = transform_bounds(
    "EPSG:4326",
    OUTPUT_CRS,
    *taipei_bbox_lonlat,
    densify_pts=21,
)

taipei_query = {
    "x": (min_x, max_x),
    "y": (min_y, max_y),
}
taipei_query

## 核心步驟：`dc.load()` 載入資料

此步驟為 ODC 工作流程之核心。查詢僅描述所需資料之條件——產品、空間範圍、
座標系統與解析度，其餘工作由 ODC 自動完成：

1. 查詢索引，篩選出與範圍相交之所有檔案；
2. 僅讀取各檔案中必要之部分，而非載入整檔；
3. 裁切並對齊至指定網格；
4. 將不同時間之資料**堆疊為具 `time` 維度之 xarray Dataset**。

相較於傳統作法（逐一開啟檔案、裁切、對齊網格、手動堆疊），此介面大幅簡化分析流程。
更重要者：當產品下之資料量自 4 個檔案增加至數千個影像切片時，
**查詢程式碼無須任何修改**——索引機制將自動篩除不相關之檔案。
此即資料立方架構之核心價值。

In [ ]:
taipei = dc.load(
    product=PRODUCT,
    measurements=[MEASUREMENT],
    x=taipei_query["x"],
    y=taipei_query["y"],
    crs=OUTPUT_CRS,
    output_crs=OUTPUT_CRS,
    resolution=RESOLUTION,
)
taipei

### 載入結果之資料結構：xarray

`dc.load()` 回傳 `xarray.Dataset`；取出 `classification` 變數後為
`xarray.DataArray`，維度為 `(time, y, x)`，即 4 個年度 × 空間網格。
xarray 支援以座標值（而非陣列索引）選取資料，例如以 `sel(time=...)` 選取特定年度，
為 Python 地球科學生態系之標準資料結構。

In [ ]:
classification = taipei[MEASUREMENT]
classification

## Area Summary

This avoids converting the full raster to a pandas Series. It counts each class directly with NumPy, then creates a small summary table.

### 中文說明

具時間維度之陣列使跨年度分析簡化為陣列運算：逐年統計各地覆類別之像元數，
乘以單一像元面積（10 m × 10 m = 100 m²）換算為平方公里。
由結果表可觀察大台北地區各年度建成區、樹林、水體等面積之變化。

In [ ]:
class_names = {
    1: "Water",
    2: "Trees",
    4: "Flooded Vegetation",
    5: "Crops",
    7: "Built Area",
    8: "Bare Ground",
    9: "Snow/Ice",
    10: "Clouds",
    11: "Rangeland",
}

pixel_area_m2 = 100
class_codes = list(class_names)

rows = []
for time_value in classification.time.values:
    arr = classification.sel(time=time_value).values
    for code in class_codes:
        pixel_count = int(np.count_nonzero(arr == code))
        rows.append(
            {
                "time": str(time_value)[:10],
                "class_code": code,
                "class_name": class_names[code],
                "pixel_count": pixel_count,
                "area_km2": pixel_count * pixel_area_m2 / 1_000_000,
            }
        )

area = pd.DataFrame(rows)
area

## Plot

The plot uses one time slice and decimates pixels for display only. The raster is still loaded in EPSG:32651, but axis tick labels are converted back to longitude and latitude.

### 中文說明

將指定年度之分類結果繪製為地圖：各地覆類別依官方配色上色、NoData 設為透明、
座標刻度轉換為經緯度。`display_step` 僅為繪圖之取樣間隔，
不影響前述統計之完整解析度。

In [ ]:
class_colors = {
    1: "#419BDF",
    2: "#397D49",
    4: "#7A87C6",
    5: "#E49635",
    7: "#C4281B",
    8: "#A59B8F",
    9: "#B39FE1",
    10: "#FFFFFF",
    11: "#E3E2C3",
}

codes = list(class_colors)
bounds = [0.5, 1.5, 2.5, 4.5, 5.5, 7.5, 8.5, 9.5, 10.5, 11.5]
cmap = ListedColormap([class_colors[code] for code in codes])
norm = BoundaryNorm(bounds, cmap.N)


def set_lonlat_ticks(ax, x_values, y_values, x_count=5, y_count=5):
    x_ticks = np.linspace(float(x_values.min()), float(x_values.max()), x_count)
    y_ticks = np.linspace(float(y_values.min()), float(y_values.max()), y_count)
    center_x = float(x_values.mean())
    center_y = float(y_values.mean())

    lon_labels, _ = transform(OUTPUT_CRS, "EPSG:4326", x_ticks.tolist(), [center_y] * len(x_ticks))
    _, lat_labels = transform(OUTPUT_CRS, "EPSG:4326", [center_x] * len(y_ticks), y_ticks.tolist())

    ax.set_xticks(x_ticks)
    ax.set_yticks(y_ticks)
    ax.set_xticklabels([f"{lon:.2f}" for lon in lon_labels])
    ax.set_yticklabels([f"{lat:.2f}" for lat in lat_labels])


plot_time_index = 0
display_step = 10

plot_data = classification.isel(time=plot_time_index)
plot_data = plot_data.where(plot_data != 0)
plot_data = plot_data.isel(y=slice(None, None, display_step), x=slice(None, None, display_step))

fig, ax = plt.subplots(figsize=(10, 9))
plot_data.plot.imshow(ax=ax, cmap=cmap, norm=norm, add_colorbar=False)
set_lonlat_ticks(ax, plot_data.x, plot_data.y)

time_value = str(classification.time.values[plot_time_index])[:10]
ax.set_title(f"Greater Taipei Land Cover - {time_value}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

patches = [
    mpatches.Patch(color=class_colors[code], label=f"{code} {class_names[code]}")
    for code in codes
]
ax.legend(handles=patches, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
plt.show()

## 小結

本教學完整示範資料立方之生命週期：

1. **建置**：PostgreSQL 索引資料庫、`datacube` 套件與實體資料檔；
2. **匯入**：註冊產品定義、為資料建立索引（僅記錄 metadata，不複製資料）；
3. **使用**：`Datacube()` 連線 → 查詢產品與資料集 → `dc.load()` 取得分析就緒之 xarray → 統計與視覺化。

Colab 環境為暫時性（runtime 回收後索引即失效），適用於教學展示；
正式部署請參見 repo 內之 [`odc_local_demo`](https://github.com/NTU-CompHydroMet-Lab/Datacube-demo/tree/main/ODC/odc_local_demo)
（Docker + 常駐 PostgreSQL，索引僅需建置一次即可持續使用，並提供 FastAPI + Leaflet 之互動網頁地圖——供一般使用者無須撰寫程式即可切換年度、勾選地覆類別並檢視面積統計）。

延伸練習：

- 修改 `taipei_bbox_lonlat` 查詢其他地區（資料涵蓋全台）；
- 依 `odc_local_demo` README 之檔名規則置入自訂 GeoTIFF，建立自訂產品與索引。